# Function 7: Sometimes Lazy is Best
You are now optimising six hyper-parameters of a machine learning model. Note that it is a popular and frequently used model, so maybe you could search to see if anyone else has optisized it before?

In [10]:
import numpy as np
from scipy.optimize import minimize
from scipy.stats import norm
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel as C

In [11]:
def load_inputs(file_path):
    with open(file_path, "r") as f:
        content = f.read()

    # Make the file a proper list of lists
    content = "[" + content.replace("]\n[", "],[") + "]"

    # Safe eval with restricted globals
    return eval(content, {"array": np.array})


def load_outputs(file_path):
    with open(file_path, "r") as f:
        content = f.read()

    content = "[" + content.replace("]\n[", "],[") + "]"

    return eval(content, {"np": np})


def get_input_points(function_number, file_path="../inputs.txt"):
    if not 1 <= function_number <= 8:
        raise ValueError("Function number must be between 1 and 8")

    data = load_inputs(file_path)
    index = function_number - 1

    inputs = [dataset[index] for dataset in data]
    return np.array(inputs)


def get_output_points(function_number, file_path="../outputs.txt"):
    if not 1 <= function_number <= 8:
        raise ValueError("Function number must be between 1 and 8")

    data = load_outputs(file_path)
    index = function_number - 1

    outputs = [row[index] for row in data]
    return np.array(outputs)

# Load inputs
X = np.load(r'initial_inputs.npy')
y = np.load(r'initial_outputs.npy')


# Get input and outputs from submissions
inputs_array = get_input_points(7)
outputs_array = get_output_points(7)


# Append inputs_f1_array to X
X = np.vstack((X, inputs_array))

# Append outputs_f1_array to Y
y = np.hstack((y, outputs_array))
y = y.ravel()

print("New shape of X:", X.shape)
print("New shape of Y:", y.shape)

New shape of X: (42, 6)
New shape of Y: (42,)


In [12]:
# Fit GP on the combined seed data and prior submissions.
kernel = C(1.0, (1e-3, 1e5)) * Matern(
    length_scale=np.ones(X.shape[1]),
    length_scale_bounds=(1e-2, 20.0),
    nu=2.5,
) + WhiteKernel(noise_level=1e-6, noise_level_bounds=(1e-8, 1e-2))
gp = GaussianProcessRegressor(
    kernel=kernel,
    n_restarts_optimizer=10,
    normalize_y=True,
    random_state=42,
)
gp.fit(X, y)

rng = np.random.default_rng(42)
best_idx = np.argmax(y)
best_point = X[best_idx]
print("Current best point:", best_point)
print("Current best output:", y[best_idx])

# Keep the search close to the strongest observed region.
top_indices = np.argsort(y)[-3:]
top_points = X[top_indices]
radius = 0.025
local_clouds = []
for point in top_points:
    local_clouds.append(point + rng.uniform(-radius, radius, size=(4000, X.shape[1])))

X_candidates = np.vstack(local_clouds)
X_candidates = np.clip(X_candidates, 0.01, 0.99)

# Mild UCB keeps some uncertainty awareness without jumping far away.
mean, std = gp.predict(X_candidates, return_std=True)
score = mean + 0.15 * std
next_query = X_candidates[np.argmax(score)]

next_query = np.round(next_query, 6)
formatted_next_query = f"{next_query[0]:.6f}-{next_query[1]:.6f}-{next_query[2]:.6f}-{next_query[3]:.6f}-{next_query[4]:.6f}-{next_query[5]:.6f}"
print("Next Query Point:", formatted_next_query)


Current best point: [0.034322 0.473025 0.245982 0.195234 0.396492 0.735091]
Current best output: 1.441511065284462
Next Query Point: 0.010749-0.454377-0.244542-0.172349-0.372556-0.739212


/opt/anaconda3/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 2 of parameter k1__k2__length_scale is close to the specified upper bound 20.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
